In [27]:
import pandas as pd
import numpy as np
import os
import time
import pickle
from matplotlib import pyplot as plt
#os.chdir('/mnt/home/icb/daniel.garger/')
data_dir='./TTP prediction/data'

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [2]:
os.getcwd()
os.chdir(os.getcwd())
os.getcwd()

'/data/gpfs/projects/punim2121/C-Path/outcome_prediction/outcome_prediction'

#### LOAD DATASETS NECESSARY TO MERGE ALL THE VARIABLES INTO ONE DATAFRAME FOR EACH PATIENT

In [28]:
variables_df=pd.read_excel('../data/variables_df.xlsx',index_col=0)
ds_names=variables_df.columns.tolist()
ttp_temporal_pat_regimens=pd.read_csv('../data/out_temporal_pat_regimens_1018_20_21_22_30.csv.gz',low_memory=False,index_col=0)

#### LOAD VARIABLES TO CONSIDER IN ANALYSIS

In [29]:
#load all the variables per patient dataframe
variables_per_patient_all=pd.read_csv('../data/all_pat_variables.csv.gz',index_col=0,low_memory=False)

#load patient IDs who are considered in this  analysis
pat_id_df=pd.read_csv('../data/patients_in_analysis.csv.gz',index_col=0)

# get all pat ids
all_ids=pat_id_df['USUBJID'].to_list()

## Load variables that are considered for the analysis for each phase
#  In 5.2_load_data_into_nested_dicts.ipnyb the variables to be considered for each phase have been selected
#  I selected the variables that are available at least for 100 patients. 
with open('../data/vars_for_analysis_per_phase.pkl', 'rb') as file:
    vars_for_analysis_per_phase = pickle.load(file)


##### **CREATE A DICTIONARY WHICH CONTAINS THE AXIS NAMES (COLUMN AND INDEX NAMES) OF THE DATAFRAMES THAT CONTAIN THE VALUES OF THE VARIABLES**
* COL_NAMES: EXTRACT THESE COLUMNS FROM THE BULK DATAFRAMES CONTAINED IN THE NESTED DICTIONARIES, THEY CONTAIN THE RELEVANT INFORMATION REGARDING THE GIVEN VARIABLE 
* INDEX_NAMES: SET THE SOLUMNS AS INDEX FOR THE DATAFRAMES

In [30]:
## As some dataset-types already have previously created dataframes contaning their results in a temporal manner (cm,ae,mh) 
# don't consider these for the merger, they will be merged to the final dataframe selected for ML in a later step 

ax_names={  'ttp_mb_dict':{'col_names':['USUBJID','STUDYID','MBDY_estimated','STD_NUM_RESULT','STD_NUM_UNIT',#'STD_CAT_RESULT','STD_CAT_UNITS',
                                        'STD_RESULT','CULTURE_STATUS'],
                         'index_names':['MBDY_estimated','USUBJID','STUDYID'],
                          'num_result':'STD_NUM_RESULT',
                          'cat_result':'STD_CAT_RESULT',
                          'day_column':'MBDY_estimated'},
            'ttp_dm_dict':{'col_names':['AGE','SEX','RACE','ARM'],
                         'index_names':['USUBJID','STUDYID']},
            #'ttp_pc_dict':{'col_names':['USUBJID','STUDYID','PCDY','PCTEST','hour','PCTPTNUM','STD_NUM_RESULT','STD_UNITS','PCSTRESC'],
            #             'index_names':['PCDY','USUBJID','STUDYID'],
            #              'num_result':'STD_NUM_RESULT',
            #              'cat_result':'PCSTRESC',
            #              'day_column':'PCDY'},
            #'ttp_pc_peaks_dict':{'col_names':['USUBJID','STUDYID','PCDY','PCTEST','hour','PCTPTNUM','STD_NUM_RESULT','STD_UNITS','PCSTRESC'],
            #             'index_names':['PCDY','USUBJID','STUDYID'],
            #              'num_result':'STD_NUM_RESULT',
            #              'cat_result':'PCSTRESC',
            #              'day_column':'PCDY'},
            #'ttp_pc_baseline_dict':{'col_names':['USUBJID','STUDYID','PCDY','PCTEST','hour','PCTPTNUM','STD_NUM_RESULT','STD_UNITS','PCSTRESC'],
            #             'index_names':['PCDY','USUBJID','STUDYID'],
            #              'num_result':'STD_NUM_RESULT',
            #              'cat_result':'PCSTRESC',
            #              'day_column':'PCDY'},
            'ttp_re_dict':{'col_names':['USUBJID','STUDYID','STD_REDY','STD_NUM_RESULT','STD_CAT_RESULT','STD_CAT_ORDINAL_RESULT',\
                                        'STD_LEFT','STD_RIGHT','STD_UNILATERAL','STD_BILATERAL'],
                         'index_names':['STD_REDY','USUBJID','STUDYID']},
            'ttp_vs_dict':{'col_names':['USUBJID','STUDYID','VSDY','STD_NUM_UNIT','STD_NUM_RESULT'],
                         'index_names':['VSDY','USUBJID','STUDYID'],
                          'num_result':'STD_NUM_RESULT',
                          'cat_result':'VSSTRESC',
                          'day_column':'VSDY'},
            'ttp_lb_dict':{'col_names':['USUBJID','STUDYID','STD_LBDY','STD_NUM_TEST','STD_NUM_RESULT','STD_NUM_UNIT','STD_NUM_RESULT_SCALED',
                                        'STD_CAT_TEST','STD_CAT_RESULT','STD_CAT_ORDINAL_RESULT'],
                                'index_names':['STD_LBDY','USUBJID','STUDYID'],
                                'num_result':'STD_NUM_RESULT_SCALED',
                                'cat_result':'STD_CAT_RESULT',
                                'day_column':'STD_LBDY'},
            #'ttp_mh_dict':{'col_names':['USUBJID','STUDYID','MHDY','STD_MHTERM','MHOCCUR'],
            #             'index_names':['MBDY','USUBJID','STUDYID']},
            'ttp_ce_dict':{'col_names':['USUBJID','STUDYID','STD_CEDY','STD_CAT_RESULT','STD_CAT_ORDINAL_RESULT','STD_CETOXGR'],
                         'index_names':['STD_CEDY','USUBJID','STUDYID']},
            #'ttp_cm_dict':{'col_names':['USUBJID','STUDYID','CMSTDY','CMENDY','CMDECOD','CMINDC','CMDOSE','CMDOSU','CMDOSFRQ','CMROUTE'],
            #             'index_names':['CMSTDY','CMENDY','USUBJID','STUDYID']},
            #'ttp_cm_ind_dict':{'col_names':['USUBJID','STUDYID','CMSTDY','CMENDY','CMINDC'],
            #             'index_names':['CMSTDY','CMENDY','USUBJID','STUDYID']},
            'ttp_mr_dict':{'col_names':['USUBJID','STUDYID','STD_MSDY','STD_CAT_RESULT','STD_CAT_ORDINAL_RESULT'],
                         'index_names':['STD_MSDY','USUBJID','STUDYID']},
            'ttp_mic_dict':{'col_names':['USUBJID','STUDYID','STD_MSDY','STD_CAT_RESULT','STD_CAT_ORDINAL_RESULT','STD_NUM_RESULT','MSSTRESU'],
                         'index_names':['STD_MSDY','USUBJID','STUDYID']},
            'ttp_ms_dict':{'col_names':['USUBJID','STUDYID','STD_MSDY','STD_CAT_RESULT','STD_CAT_ORDINAL_RESULT','SUSC_CONC','RESISTANCE_CONC'],
                         'index_names':['STD_MSDY','USUBJID','STUDYID']},
            #'ttp_ae_dict':{'col_names':['USUBJID','STUDYID','AEDY','AESTDY','AEENDY','STD_AETERM','AEREL'],
            #             'index_names':['AEDY','AESTDY','AEENDY','USUBJID','STUDYID']},
            'ttp_su_dict':{'col_names':['USUBJID','STUDYID','SUDY','STD_CAT_RESULT','STD_CAT_ORDINAL_RESULT'],
                         'index_names':['SUDY','USUBJID','STUDYID']}}




### Common index for the dataframe holding all the patients' data
common_index=['DAY','USUBJID','STUDYID']

#### **1. LOOP OVER EACH PHASE AND MERGE DATA FOR ALL PATIENTS IN THAT PHASE FROM ONE DATASET-TYPE (i.e. mb,dm,pc...) INTO ONE DATAFRAME**
##### FORMAT:
#### -  COLUMNS: VARNAME_COLUMNNAME (COLUMNNAMES FROM AX_NAMES DICT)
#### -  INDEX/ROWS: TIMEPOINT OF MEASUREMENT (DAY)

#### **2. ADD DRUG REGIMEN AND DEMOGRAPHIC DATAFRAME TO THE PREVIOUSLY CREATED LIST OF DATASET-TYPE DATAFRAMES**
#### **3. MERGE ALL DATAFRAMES CREATED FOR EACH DATASET-TYPE (i.e. mb,dm,pc...) INTO ONE DATAFRAME FOR EACH PHASE (phase_df)**

In [31]:
## Customize function for creating mean of numerical columns and keeping the string values when 
# taking average of multiple measurements from same day
def mean_str(col):
    if pd.api.types.is_numeric_dtype(col):
        return col.mean()
    if pd.api.types.is_numeric_dtype(col)==False or col.isna().all():
        return col.unique().tolist()[0] if col.nunique() == 1 else np.nan
        

# If multiple numerical measurements from one day, get their daily means and return as a dataframe
def get_daily_mean_of_numeric(df,ds_dict_name):
    n_res_name=ax_names[ds_dict_name]['num_result']
    day_column=ax_names[ds_dict_name]['day_column']
   
    # Take average of numerical columns, keep the string columns as is
    if df.loc[:,n_res_name].isna().all()==False:
        df=df.groupby(by=day_column).agg(mean_str).reset_index()
    return df

## Check for ambiguous result originating from the same day 
## Check if the ambiguous results are from the mb dataset -> probably from the standardised categorical mb results
def check_for_ambiguous_mb_results_from_same_day(ds,ds_name,dupl):
    ## Check if the ambiguous results are from the mb dataset -> probably from the standardised categorical mb results
    #  with the colnames name_of_test_RESULT (not _CAT_RESULT/NUM_RESULT!)
    if ds_name=='mb':
        
        ## Loop over ambiguous results of patients and check if there are mb results that are not positive or negative
        df=dupl.copy()
        
        ## Extract column names with ambig. results
        cols_with_ambig=df.columns[df.apply(lambda x:len(x.unique())>1)&\
                                            (dupl.columns.str.endswith('RESULT'))&\
                                            ~(dupl.columns.str.contains('CAT|NUM',na=False))]

        ## loop over columns that have ambig measurements from the given day 
        std_results=[]
        for ambig_col in cols_with_ambig:
            
            ## If there are 2 ambiguous measurement result values ->
            #  Set these measurements to "POSITIVE" as there is an equal number of negative/positive measurements
            if len(df.loc[:,ambig_col].mode())==2:

                ## Check if there any other results then 'negative' or 'positive' -> raise error if yes                   
                unique_mb_results=sorted(set(pd.unique(df.loc[:,ambig_col].values.ravel())))
                if unique_mb_results!=['negative', 'positive']:
                    print(df.loc[:,ambig_col])
                    raise ValueError('Invalid mb result! (not "negative" or "positive")')
                                    
                mb_result='positive'
                std_results.append(mb_result)


            ## If there is a majority of a result (neg or pos) -> take that
            if len(df.loc[:,ambig_col].mode())==1:
                unique_mb_results=sorted(set((df.loc[:,ambig_col].unique().tolist())))
                if unique_mb_results!=['negative', 'positive']:
                    print(df.loc[:,ambig_col])
                    raise ValueError('Invalid mb result! (not "negative" or "positive")')
                
                mb_result=df.loc[:,ambig_col].mode()[0]
                std_results.append(mb_result)


            ## Check if there are invalid results on top of positive negative
            if len(df.loc[:,ambig_col].mode())>2:
                unique_mb_results=sorted(set(pd.unique(df.loc[:,ambig_col].values.ravel())))
                if unique_mb_results!=['negative'] and unique_mb_results!=['positive']:
                    print(df.loc[:,ambig_col])
                    raise ValueError('Invalid mb result! (more then 2 result categories)')
            
        ## Update original dataframe with the standardised result for the given day
        #  Insert the updated results to the first of the duplicated rows and set all others to np.nan ->
        #  They will be dropped at the next drop_duplicates() function
        ds.loc[df.index,cols_with_ambig]=std_results  
        ds=ds.drop_duplicates()

        return ds
        
    else:
        print(dupl)
        print('Duplication is not in "mb" dataset, but in '+ds_name+' dataset!')
        return ds
        #raise ValueError('Duplication is not in "mb" dataset, but in '+ds_name+' dataset!')  


### SAME FUNCTION as the previous, but checks for the already concatenated ds_df dataframe, where all 
### the patients are contained -> algorithm runs faster
## Check for ambiguous result originating from the same day 
## Check if the ambiguous results are from the mb dataset -> probably from the standardised categorical mb results
def check_for_ambiguous_mb_results_from_same_day2(ds,ds_name,dupl):
    ## Check if the ambiguous results are from the mb dataset -> probably from the standardised categorical mb results
    #  with the colnames name_of_test_RESULT (not _CAT_RESULT/NUM_RESULT!)
    if ds_name=='mb':
        
        ## Loop over ambiguous results of patients and check if there are mb results that are not positive or negative
        #df=dupl.copy()
        for _,df in dupl.groupby(by=['USUBJID','DAY']):
            ## Extract column names with ambig. results
            cols_with_ambig=df.columns[df.apply(lambda x:len(x.unique())>1)&(dupl.columns.str.startswith('mb_'))&\
                                                (dupl.columns.str.endswith('RESULT'))&\
                                                ~(dupl.columns.str.contains('CAT|NUM',na=False))]

            ## loop over columns that have ambig measurements from the given day 
            std_results=[]
            for ambig_col in cols_with_ambig:
                
                ## If there are 2 ambiguous measurement result values ->
                #  Set these measurements to "POSITIVE" as there is an equal number of negative/positive measurements
                if len(df.loc[:,ambig_col].mode())==2:

                    ## Check if there any other results then 'negative' or 'positive' -> raise error if yes                   
                    unique_mb_results=sorted(set(pd.unique(df.loc[:,ambig_col].values.ravel())))
                    if unique_mb_results!=['negative', 'positive']:
                        print(df.loc[:,ambig_col])
                        raise ValueError('Invalid mb result! (not "negative" or "positive")')
                                        
                    mb_result='positive'
                    std_results.append(mb_result)


                ## If there is a majority of a result (neg or pos) -> take that
                if len(df.loc[:,ambig_col].mode())==1:
                    unique_mb_results=sorted(set((df.loc[:,ambig_col].unique().tolist())))
                    if unique_mb_results!=['negative', 'positive']:
                        print(df.loc[:,ambig_col])
                        raise ValueError('Invalid mb result! (not "negative" or "positive")')
                    
                    mb_result=df.loc[:,ambig_col].mode()[0]
                    std_results.append(mb_result)


                ## Check if there are invalid results on top of positive negative
                if len(df.loc[:,ambig_col].mode())>2:
                    unique_mb_results=sorted(set(pd.unique(df.loc[:,ambig_col].values.ravel())))
                    if unique_mb_results!=['negative'] and unique_mb_results!=['positive']:
                        print(df.loc[:,ambig_col])
                        raise ValueError('Invalid mb result! (more then 2 result categories)')
            
            ## Update original dataframe with the standardised result for the given day
            #  Insert the updated results to the first of the duplicated rows and set all others to np.nan ->
            #  They will be dropped at the next drop_duplicates() function
            ds.loc[df.index,cols_with_ambig]=std_results  
            #ds=ds.drop_duplicates()

        return ds
        
    else:
        print(dupl)
        print('Duplication is not in "mb" dataset, but in '+ds_name+' dataset!')
        return ds
        #raise ValueError('Duplication is not in "mb" dataset, but in '+ds_name+' dataset!')  







## Function for merging data for one dataset type for all patients
def merge_all_pats_data_for_ds_type(pat_ids,ds_dict,ds_name,vars_for_analysis,dm_df):
    from pandas.errors import InvalidIndexError

    ds_dict_name='ttp_'+ds_name+'_dict'

    #initialize ds_df:
    #ds_df: df containing results of all the variables of A SINGLE DATASET-TYPE (mb,dm...) FOR ALL PATIENTS
    n=len(common_index)
    mult_ind=pd.MultiIndex(levels=[[]]*n,codes=[[]]*n,names=common_index)
    ds_df=pd.DataFrame(index=mult_ind)
    ds_df_list=[]

    num=0
    num_allpat=len(pat_ids)
    start = time.time()

    for id in pat_ids:
        num=num+1
        print(round(num/num_allpat*100,3),'%',end="")
        print("\r", end="")
        
        try:
            pat_ds_dict=ds_dict[id]                
        except KeyError:
            continue

        var_df_list=[]

        # select only variables in the dataframe that are considered for this analysis 
        # (drop the variables that don't have anough timepoints)
        ds_vars_for_analysis=list(set(pat_ds_dict['variables'])&set(vars_for_analysis))
        #print(ds_vars_for_analysis)

        # Only create dataframe of a variable if it is contained in the list of considered variables
        if len(ds_vars_for_analysis)>0:

            # Iterate through the selected variables of one patient->create a df with results for one variable 
            # and append them to var_df_list
            for var in ds_vars_for_analysis:
                df=pat_ds_dict['grouped_data'].get_group(var)
                df=df.drop_duplicates()
                df.columns=df.columns.str.replace("STD_NUM_UNITS|VSSTRESU", "STD_NUM_UNIT", regex=True)
                df.columns=df.columns.str.replace("VSSTRESN", "STD_NUM_RESULT", regex=True)
                #print( df.columns.tolist())
                important_colnames=list(set(df.columns.tolist())&(set(ax_names[ds_dict_name]['col_names'])))
                df=df.loc[:,important_colnames]

                            
                #check if results are numerical -> then get the daily means of the numerical measurements
                if 'num_result' in ax_names[ds_dict_name].keys():
                    df=get_daily_mean_of_numeric(df,ds_dict_name)
                            
                ## Check for duplicated indices -> can be a sign for ambiguous microbiological results originating from the same day
                #  If there are duplicates coming from ambig. results ->standardise them to a unique result per day 
                #  either by majority vote or if the nummber of negative/positive mb results are equal, set result to positive
                
                df=df.drop_duplicates()
                #dupl=df[df[ax_names[ds_dict_name]['index_names']].duplicated(keep=False)]
                #if len(dupl)>0:
                #    df=check_for_ambiguous_mb_results_from_same_day(df,ds_name,dupl)

                ## Update inedex to multiindex and add to identify which column name is referring 
                #  to which variable, concatenate 'var_' in the front of column namesfrom ax_names dict 
                #  (i.e. var=pc_Bedaquiline, PCTEST->pc_Bedaquiline_PCTEST)
                df=df.set_index(keys=ax_names[ds_dict_name]['index_names'],drop=True)
                update_cols=[var+'_'+x for x in df.columns]
                df.columns=update_cols
                var_df_list.append(df)

            # initialize var_df with same multiindex created in previuos step
            n=len(ax_names[ds_dict_name]['index_names'])
            mult_ind=pd.MultiIndex(levels=[[]]*n,codes=[[]]*n,names=ax_names[ds_dict_name]['index_names'])
            var_df=pd.DataFrame(index=mult_ind)

            #var_df: df containing results of all the variables of ONE DATASET-TYPE (i.e.mb, dm,...) PER PATIENT
            for d in var_df_list:
                var_df=pd.merge(var_df,d,left_index=True,right_index=True,how='outer')
            
            var_df.index.rename('DAY',level=0,inplace=True)         
            
            #var_df_list.append(var_df)
            #var_df=pd.concat(var_df_list,axis=1)                                  
            
                            
            ## If height is available for patient, add it to dataframe, as some studies lack the day of height measurement
            #  therefore are not captured by the previous functions
            #if id in height_df['USUBJID'].unique() and ds_name=='vs':
            #    height=height_df.loc[height_df['USUBJID']==id,'STD_NUM_RESULT_SCALED'].values
            #    var_df.loc[:,'vs_Height_STD_NUM_RESULT_SCALED']=[height,]*len(var_df)

            ds_df_list.append(var_df)


    ## Append ds_df to the list and merge all previously created dataframes of the patients into one dataframe                
    ds_df_list.append(ds_df)
    ds_df=pd.concat(ds_df_list,axis=0)    

      
    #print('len(ds_df_list)',len(ds_df_list))
    
    dupl=ds_df[ds_df.index.duplicated(keep=False)]
    if len(dupl)>0:
        ds_df=check_for_ambiguous_mb_results_from_same_day2(ds_df,ds_name,dupl)

    ## Drop duplicates from the same day and patient and set the index again to the common multiindex
    ds_df=ds_df.reset_index()    
    ds_df=ds_df.drop_duplicates()
    ds_df=ds_df.set_index(['USUBJID','DAY','STUDYID']) 

    ## Standardise vs numerical column names
    #if ds_name=='vs':
    #    ds_df.columns=ds_df.columns.str.replace('VSSTRESU','STD_NUM_UNIT')

    end = time.time()
    print(round((end - start)/60,3),'minutes runtime')
    

    return ds_df
    


## For each dataset-type merge all data for the patients within a dataset type

- ##### **1. LOOP OVER EACH PHASE AND MERGE DATA FOR ALL PATIENTS IN THAT PHASE FROM ONE DATASET-TYPE (i.e. mb,dm,pc...) INTO ONE DATAFRAME**
- ##### FORMAT:
   - ##### -  COLUMNS: VARNAME_COLUMNNAME (COLUMNNAMES FROM AX_NAMES DICT)
   - ##### -  INDEX/ROWS: TIMEPOINT OF MEASUREMENT (DAY)
   - ##### -  SAVE MERGED DATAFRAMES FOR EACH DATASET TYPE AS CSV.GZ ==> THIS WAY IF ONLY THE GIVEN DATASET TYPE DATAFRAME NEEDS TO BE UPDATED IF NECESSARY

In [32]:
dm=pd.read_csv('../data/out_dm.csv.gz',low_memory=False)
dm=dm.set_index('USUBJID',drop=True)
dm_colnames=ax_names['ttp_dm_dict']['col_names']


## Load height data of patients
weight_height_df=pd.read_csv('../data/weight_height_bmi_of_patients.csv.gz',index_col=0,low_memory=False)
height_df=weight_height_df[weight_height_df['VSTEST']=='Height']

## Make list of dataset-types to loop over. As some dataset-types already have previously created dataframes contaning 
#  their results in a temporal manner (cm,ae,mh) don't consider these for the merger, they will be merged to the final dataframe
#  selected for ML in a later step 
ds_names=[x.split('_')[1] for x in [*ax_names] if x!='ttp_dm_dict']

## Loop over phases and select patient in that phase 
for phase in [3]:
    phase_name='phase_'+str(phase)
    print(phase_name)
    pat_ids_in_phase=pat_id_df.loc[:,'USUBJID'].tolist()[:]    

    ## Subset ddm dataframe to patients considered in the phase
    dm_df=dm.loc[pat_ids_in_phase,dm_colnames]

 
    ## Loop over the considered dataset-types to merge 
    for ds_name in ds_names[:1]:
        print(ds_name)
        f = open('../data/out_'+ds_name+'_dict','rb')
        ds_dict=pickle.load(f)
        f.close()
        
        
        ## Create merged dataframe for the given dataset-type containing results for all patients considered in the phase
        merged_df_for_ds_type=merge_all_pats_data_for_ds_type(pat_ids_in_phase,ds_dict,ds_name,
                                                              vars_for_analysis_per_phase[phase_name],dm_df)

        ## Save merged df for ds_type of df is not empty
        if len(merged_df_for_ds_type)>0:    
            merged_df_for_ds_type.to_csv(f'../data/phase{phase}_{ds_name}_merged_df.csv.gz',compression='gzip')  

        ## Delete ds_dict to save memory 
        del ds_dict
       

phase_3
mb
1.127 minutes runtime


In [23]:
merged_df_for_ds_type.dropna(how='all',axis=0)#.columns

,,,su_SMOKING EVER_STD_CAT_ORDINAL_RESULT,su_SMOKING EVER_STD_CAT_RESULT
USUBJID,DAY,STUDYID,,
TB-1020/1016,1,TB-1020,0.0,N
TB-1020/1023,1,TB-1020,0.0,N
TB-1020/1029,1,TB-1020,1.0,Y
TB-1020/1031,1,TB-1020,0.0,N
TB-1020/1039,1,TB-1020,0.0,N
...,...,...,...,...
TB-1021/1045709,1,TB-1021,1.0,Y
TB-1021/1048641,1,TB-1021,1.0,Y
TB-1021/2811814,1,TB-1021,0.0,N


## Merge all the dataset type dataframes into one dataframe holding all the clinical data
- ##### **1. ADD DRUG REGIMEN AND DEMOGRAPHIC DATAFRAME TO THE PREVIOUSLY CREATED LIST OF DATASET-TYPE DATAFRAMES**
- ##### **2. MERGE ALL DATAFRAMES CREATED FOR EACH DATASET-TYPE (i.e. mb,dm,pc...) INTO ONE DATAFRAME FOR EACH PHASE (phase_df)**

In [33]:
dm=pd.read_csv('../data/out_dm.csv.gz',low_memory=False)
dm=dm.set_index('USUBJID',drop=True)
dm_colnames=ax_names['ttp_dm_dict']['col_names']


## Load height data of patients
weight_height_df=pd.read_csv('../data/weight_height_bmi_of_patients.csv.gz',index_col=0,low_memory=False)
height_df=weight_height_df[weight_height_df['VSTEST']=='Height']

## Make list of dataset-types to loop over. As some dataset-types already have previously created dataframes contaning 
#  their results in a temporal manner (cm,ae,mh) don't consider these for the merger, they will be merged to the final dataframe
#  selected for ML in a later step 
ds_names=[x.split('_')[1] for x in [*ax_names] if x!='ttp_dm_dict']

## Loop over phases and select patient in that phase 
for phase in [3]:
    phase_name='phase_'+str(phase)
    print(phase_name)
    pat_ids_in_phase=pat_id_df.loc[:,'USUBJID'].tolist()[:]    

    ## Subset ddm dataframe to patients considered in the phase
    dm_df=dm.loc[pat_ids_in_phase,dm_colnames]

    ## List to collect the dataset-type dataframes into
    merged_data_in_phase=[]

    ## Loop over the considered dataset-types to merge 
    for ds_name in ds_names[:]:
        print(ds_name)
        fn=f'../data/phase{phase}_{ds_name}_merged_df.csv.gz'
        if os.path.isfile(fn):
        
            merged_df_for_ds_type=pd.read_csv(f'../data/phase{phase}_{ds_name}_merged_df.csv.gz',index_col=0,low_memory=False)
            merged_df_for_ds_type=merged_df_for_ds_type.reset_index().set_index(['USUBJID','DAY','STUDYID']) 
            #print(merged_df_for_ds_type.index)
            merged_data_in_phase.append(merged_df_for_ds_type)

    
    ## Merge all previously created dataframes into one dataframe holding all the data for all patients in phase + all 
    #  dataset-types and save it
    phase_df=pd.concat(merged_data_in_phase,axis=1)
    phase_df_reset_idx=phase_df.reset_index()

    ## ADD DEMOGRAPHIC DATA OF THE PATIENTS
    for pat_id,pat_df in phase_df_reset_idx.groupby(by=['USUBJID']):
        phase_df_reset_idx.loc[pat_df.index,dm_colnames]=dm_df.loc[pat_id,dm_colnames].values

    phase_df_reset_idx.to_csv('../data/merged_df.csv.gz',compression='gzip')  
        
    #del merged_data_in_phase,phase_df,phase_df_reset_idx

phase_3
mb
re
vs
lb
ce
mr
mic
ms
su


In [26]:
phase_df_reset_idx.loc[:,phase_df_reset_idx.columns.str.startswith('mb_')]#.dropna(how='all',axis=1)

,mb_MGIT_STD_RESULT,mb_MGIT_STD_NUM_RESULT,mb_MGIT_CULTURE_STATUS,mb_MGIT_STD_NUM_UNIT,mb_ZN-smear_STD_RESULT,mb_ZN-smear_STD_NUM_RESULT,mb_ZN-smear_CULTURE_STATUS,mb_ZN-smear_STD_NUM_UNIT,mb_HAIN-test_STD_RESULT,mb_HAIN-test_STD_NUM_RESULT,...,mb_AccuProbe_CULTURE_STATUS,mb_AccuProbe_STD_NUM_UNIT,mb_MPT64-Antigen-Test_STD_RESULT,mb_MPT64-Antigen-Test_STD_NUM_RESULT,mb_MPT64-Antigen-Test_CULTURE_STATUS,mb_MPT64-Antigen-Test_STD_NUM_UNIT,mb_RT-PCR_STD_RESULT,mb_RT-PCR_STD_NUM_RESULT,mb_RT-PCR_CULTURE_STATUS,mb_RT-PCR_STD_NUM_UNIT
0,negative,NaN,NaN,NaN,negative,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,negative,NaN,NaN,NaN,negative,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,positive,NaN,NaN,NaN,negative,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,negative,NaN,NaN,NaN,negative,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,negative,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83659,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83660,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83661,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83662,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
